In [2]:
# ── LSTM.ipynb — imports, reproducibility, device ────────────────────────
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import features as F

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)          # 'mps' on Apple Silicon — much faster than cpu

SEQ_LEN  = 55
VAL_FRAC = 0.2

device: mps


In [3]:
import joblib, lightgbm as lgb
import features as F

full = F.build_features(pd.read_csv("Data/train.csv"))
feat_cols = F.feature_columns(full)
cols = feat_cols + ["stock_id"]
X = full[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = full["target"]; ok = y.notna().values

gbm = lgb.LGBMRegressor(objective="mae", n_estimators=500, learning_rate=0.03,
                        num_leaves=31, min_child_samples=1000, reg_alpha=1.0, reg_lambda=1.0,
                        subsample=0.7, subsample_freq=1, colsample_bytree=0.7,
                        random_state=42, n_jobs=-1, verbose=-1)
gbm.fit(X[ok], y[ok], categorical_feature=["stock_id"])

joblib.dump({"model": gbm, "feat_cols": feat_cols}, "lgb_final.joblib")
print("saved lgb_final.joblib")

saved lgb_final.joblib
